In [4]:
import sys
from pathlib import Path

print("Python executable:")
print(sys.executable)

print("\nWorking directory:")
print(Path.cwd())

Python executable:
/var/home/connor/Python/$ Projects/materials-pipeline/.venv/bin/python

Working directory:
/var/home/connor/Python/$ Projects/materials-pipeline/notebooks


In [5]:
import importlib.util

package = importlib.util.find_spec("materials_pipeline")

print(package)

ModuleSpec(name='materials_pipeline', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7f6f14824470>, origin='/var/home/connor/Python/$ Projects/materials-pipeline/src/materials_pipeline/__init__.py', submodule_search_locations=['/var/home/connor/Python/$ Projects/materials-pipeline/src/materials_pipeline'])


In [6]:
from datetime import UTC, datetime

from materials_pipeline.models import (
    CompositionRecord,
    MaterialRecord,
    ProviderName,
    SourceReference,
)

In [7]:
source = SourceReference(
    provider=ProviderName.MATERIALS_PROJECT,
    source_id="mp-149",
    retrieved_at=datetime.now(UTC),
    source_url="https://materialsproject.org/materials/mp-149",
)

composition = CompositionRecord(
    reduced_formula="Si",
    chemical_system="Si",
    element_amounts={"Si": 1.0},
)

material = MaterialRecord(
    composition=composition,
    sources=[source],
)

material

MaterialRecord(record_id=UUID('c662a926-f669-4858-9e42-699e4e2b5efe'), composition=CompositionRecord(reduced_formula='Si', chemical_system='Si', element_amounts={'Si': 1.0}), structure=None, sources=[SourceReference(provider=<ProviderName.MATERIALS_PROJECT: 'materials_project'>, source_id='mp-149', retrieved_at=datetime.datetime(2026, 8, 28, 12, 23, 26, 878560, tzinfo=datetime.timezone.utc), source_url='https://materialsproject.org/materials/mp-149')])

Serialise and dump the material to produce JSON

In [8]:
material.model_dump(mode="json")
print(material.model_dump_json(indent=2))

{
  "record_id": "c662a926-f669-4858-9e42-699e4e2b5efe",
  "composition": {
    "reduced_formula": "Si",
    "chemical_system": "Si",
    "element_amounts": {
      "Si": 1.0
    }
  },
  "structure": null,
  "sources": [
    {
      "provider": "materials_project",
      "source_id": "mp-149",
      "retrieved_at": "2026-08-28T12:23:26.878560Z",
      "source_url": "https://materialsproject.org/materials/mp-149"
    }
  ]
}


## Test validations

In [ ]:
from pydantic import ValidationError

try:
    invalid_source = SourceReference(
        provider='unknown_provider',
        source_id='example-001',
        retrieved_at=datetime.now(UTC),
    )
except ValidationError as exc:
    print(exc)

1 validation error for SourceReference
provider
  Input should be 'materials_project', 'nomad' or 'csd' [type=enum, input_value='unknown_provider', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/enum


This should fils because "unknown_provider" isn't one of the values permitted by `ProviderName`

In [ ]:
try:
    invalid_composition = CompositionRecord(
        reduced_formula='Si',
        chemical_system='Si',
        element_amounts={'Si': 'not a number'},
    )
except ValidationError as exc:
    print(exc)

1 validation error for CompositionRecord
element_amounts.Si
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='not a number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing


This should fils because the element amount cannot be interpreted as a floating-point value.

## A basic materials adapter

In [ ]:
# Mock Materials Project record
mp_record = {
    'material_id': 'mp-149',
    'formula_pretty': 'Si',
    'chemsys': 'Si',
    'structure': None,
}

# Provider field       Internal field
# --------------       --------------
# material_id          sources[].source_id
# formula_pretty       composition.reduced_formula
# chemsys              composition.chemical_system
# structure            structure

In [12]:
from materials_pipeline.providers.materials_project import MaterialsProjectProvider

# Create the adapter
provider = MaterialsProjectProvider()

# Transform the mock input
material = provider.transform(mp_record)

material

MaterialRecord(record_id=UUID('b6de7d78-0c91-47e9-9ede-706c2872443a'), composition=CompositionRecord(reduced_formula='Si', chemical_system='Si', element_amounts={'Si': 1.0}), structure=None, sources=[SourceReference(provider=<ProviderName.MATERIALS_PROJECT: 'materials_project'>, source_id='mp-149', retrieved_at=datetime.datetime(2026, 8, 28, 12, 23, 27, 52790, tzinfo=datetime.timezone.utc), source_url='https://materialsproject.org/materials/mp-149')])

In [13]:
# Inspect the resulting JSON
print(material.model_dump_json(indent=2))

{
  "record_id": "b6de7d78-0c91-47e9-9ede-706c2872443a",
  "composition": {
    "reduced_formula": "Si",
    "chemical_system": "Si",
    "element_amounts": {
      "Si": 1.0
    }
  },
  "structure": null,
  "sources": [
    {
      "provider": "materials_project",
      "source_id": "mp-149",
      "retrieved_at": "2026-08-28T12:23:27.052790Z",
      "source_url": "https://materialsproject.org/materials/mp-149"
    }
  ]
}


In [ ]:
# A deliberately unreduced formula
unreduced_record = {
    'material_id': 'example-001',
    'formula_pretty': 'Si2O4',
    'chemsys': 'Si-O',
    'structure': None,
}

normalised = provider.transform(unreduced_record)

print(normalised.model_dump_json(indent=2))

{
  "record_id": "72a78e1c-4d87-4347-88c2-9ec91a605c6f",
  "composition": {
    "reduced_formula": "SiO2",
    "chemical_system": "O-Si",
    "element_amounts": {
      "Si": 1.0,
      "O": 2.0
    }
  },
  "structure": null,
  "sources": [
    {
      "provider": "materials_project",
      "source_id": "example-001",
      "retrieved_at": "2026-08-28T12:23:27.067590Z",
      "source_url": "https://materialsproject.org/materials/example-001"
    }
  ]
}
